# FIMMOGA Project — Clean Reproducible Notebook

This notebook consolidates the project code used for the report:

- Datasets: **Cora**, **Email-Eu-core**, **Facebook**, **CollegeMsg**, **OpenFlights-NA**
- Diffusion model: **Independent Cascade (IC)**
- Algorithms:
  - Heuristic baselines (Degree / Closeness / Betweenness / PageRank, Per-Group Top-Degree)
  - RIS-Greedy baseline
  - **CELF++ (IC)** baseline (real implementation; no stub)
  - **FIMMOGA (orig)**: MC-in-GA, 2 objectives (Spread + 1−Gini), **NSGA-II**
  - **FIMMOGA++ (proposed)**: RIS-in-GA, 5 objectives (Spread, 1−Gini, Max–Min, 1−Parity, Welfare), **NSGA-III**

**Directory layout expected (project root):**
- `data_cora/` containing `cora.content`, `cora.cites`
- `data_email_eu_core/` containing `email-Eu-core.txt`, `email-Eu-core-department-labels.txt`
- `data_facebook/` containing `facebook_combined.txt`
- `data_collegemsg/` containing `CollegeMsg.txt.gz`
- `data_openflights/` containing `airports.dat`, `routes.dat`

> Tip: Most “report tables” use **k=20, p=0.10** and run a larger MC evaluation at the end.


In [ ]:
# ==== 1) Imports & global config (reproducibility) ====
import os, time, math, random, gzip, csv, heapq
from pathlib import Path
from collections import defaultdict, Counter
from dataclasses import dataclass
from typing import Dict, List, Set, Tuple, Iterable, Optional

import numpy as np
import pandas as pd
import networkx as nx

def _get_tqdm():
    try:
        from tqdm.auto import tqdm
        return tqdm
    except Exception:
        return None

SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

PROJECT_ROOT = Path('.').resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULTS_DIR  =', RESULTS_DIR)

## 2) Graph utilities

In [ ]:

# ==== Graph utilities
def to_undirected_nx(edges: Iterable[Tuple[int,int]]) -> nx.Graph:
    G = nx.Graph()
    G.add_edges_from((int(u), int(v)) for (u, v) in edges if int(u) != int(v))
    return G

def graph_stats(G: nx.Graph) -> Tuple[int,int]:
    return G.number_of_nodes(), G.number_of_edges()

def remap_to_consecutive_ids(G: nx.Graph):
    mapping = {u:i for i,u in enumerate(G.nodes())}
    invmap = {i:u for u,i in mapping.items()}
    H = nx.relabel_nodes(G, mapping, copy=True)
    return H, mapping, invmap

def build_adj_list(G: nx.Graph) -> Dict[int, List[int]]:
    return {u: list(G.neighbors(u)) for u in G.nodes()}

def community_groups(G: nx.Graph, method="greedy") -> Dict[int, int]:
    if method == "greedy":
        comms = list(nx.algorithms.community.greedy_modularity_communities(G))
    elif method == "label_prop":
        comms = list(nx.algorithms.community.asyn_lpa_communities(G, seed=SEED))
    else:
        raise ValueError("Unknown community method.")
    group_of = {}
    for gid, cset in enumerate(comms):
        for u in cset:
            group_of[int(u)] = gid
    return group_of

def ensure_groups(G: nx.Graph, group_of: Optional[Dict[int,int]]) -> Dict[int,int]:
    if group_of is None:
        return community_groups(G, method="greedy")
    fallback_gid = max(group_of.values())+1 if len(group_of)>0 else 0
    out = {}
    for u in G.nodes():
        out[u] = int(group_of.get(u, fallback_gid))
    return out


## 3) Dataset loaders (local folders in project root)

In [ ]:
# ==== Dataset loaders (local-only; no internet) ====
def take_largest_cc(G: nx.Graph, group_of: Dict[int,int]):
    if G.number_of_nodes() == 0:
        return G, group_of
    largest = max(nx.connected_components(G), key=len)
    G2 = G.subgraph(largest).copy()
    group2 = {u: group_of[u] for u in G2.nodes()}
    return G2, group2

def load_cora(data_dir=PROJECT_ROOT/'data_cora'):
    content = Path(data_dir) / 'cora.content'
    cites   = Path(data_dir) / 'cora.cites'
    if not content.exists() or not cites.exists():
        raise FileNotFoundError(f'Missing Cora files in {data_dir}')

    id_to_idx = {}
    labels = {}
    with open(content, 'r') as f:
        for line in f:
            parts = line.strip().split()
            pid = parts[0]
            lab = parts[-1]
            if pid not in id_to_idx:
                id_to_idx[pid] = len(id_to_idx)
            labels[id_to_idx[pid]] = int(lab) if lab.isdigit() else lab

    G = nx.Graph()
    G.add_nodes_from(labels.keys())
    with open(cites, 'r') as f:
        for line in f:
            a, b = line.strip().split()
            if a in id_to_idx and b in id_to_idx:
                u, v = id_to_idx[a], id_to_idx[b]
                if u != v:
                    G.add_edge(u, v)

    group_of = {u: int(labels[u]) if isinstance(labels[u], (int, np.integer)) else labels[u] for u in labels}
    G, group_of = take_largest_cc(G, group_of)
    return G, group_of, 'Cora'

def load_email_eu_core(data_dir=PROJECT_ROOT/'data_email_eu_core'):
    edges_f = Path(data_dir) / 'email-Eu-core.txt'
    labs_f  = Path(data_dir) / 'email-Eu-core-department-labels.txt'
    if not edges_f.exists() or not labs_f.exists():
        raise FileNotFoundError(f'Missing Email-Eu-core files in {data_dir}')

    group_of = {}
    with open(labs_f, 'r') as f:
        for line in f:
            node, dept = line.strip().split()
            group_of[int(node)] = int(dept)

    G = nx.Graph()
    with open(edges_f, 'r') as f:
        for line in f:
            u, v = map(int, line.strip().split())
            if u != v:
                G.add_edge(u, v)

    # keep labeled nodes only (fairness needs groups)
    labeled_nodes = [u for u in G.nodes() if u in group_of]
    G = G.subgraph(labeled_nodes).copy()
    group_of = {u: group_of[u] for u in G.nodes()}
    G, group_of = take_largest_cc(G, group_of)
    return G, group_of, 'Email-Eu-core'

def load_facebook(data_dir=PROJECT_ROOT/'data_facebook'):
    edges_f = Path(data_dir) / 'facebook_combined.txt'
    if not edges_f.exists():
        raise FileNotFoundError(f'Missing facebook_combined.txt in {data_dir}')
    G = nx.Graph()
    with open(edges_f, 'r') as f:
        for line in f:
            u, v = map(int, line.strip().split())
            if u != v:
                G.add_edge(u, v)

    # use largest CC and greedy modularity communities as groups
    dummy = {u: 0 for u in G.nodes()}
    G, _ = take_largest_cc(G, dummy)
    comms = list(nx.algorithms.community.greedy_modularity_communities(G))
    group_of = {}
    for gi, cset in enumerate(comms):
        for u in cset:
            group_of[u] = gi
    return G, group_of, 'Facebook'

def load_collegemsg(data_dir=PROJECT_ROOT/'data_collegemsg'):
    gz_f = Path(data_dir) / 'CollegeMsg.txt.gz'
    if not gz_f.exists():
        raise FileNotFoundError(f'Missing CollegeMsg.txt.gz in {data_dir}')
    G = nx.Graph()
    with gzip.open(gz_f, 'rt') as f:
        for line in f:
            if not line.strip():
                continue
            u, v = map(int, line.strip().split()[:2])
            if u != v:
                G.add_edge(u, v)

    dummy = {u: 0 for u in G.nodes()}
    G, _ = take_largest_cc(G, dummy)
    comms = list(nx.algorithms.community.greedy_modularity_communities(G))
    group_of = {}
    for gi, cset in enumerate(comms):
        for u in cset:
            group_of[u] = gi
    return G, group_of, 'CollegeMsg'

def load_openflights_na(data_dir=PROJECT_ROOT/'data_openflights'):
    airports_f = Path(data_dir) / 'airports.dat'
    routes_f   = Path(data_dir) / 'routes.dat'
    if not airports_f.exists() or not routes_f.exists():
        raise FileNotFoundError(f'Missing OpenFlights files in {data_dir}')

    airports = {}
    with open(airports_f, 'r', encoding='utf-8', errors='ignore') as f:
        rdr = csv.reader(f)
        for row in rdr:
            try:
                aid = int(row[0])
                country = row[3].strip()
                lat = float(row[6]); lon = float(row[7])
                airports[aid] = (lat, lon, country)
            except Exception:
                continue

    def in_na(lat, lon):
        return (5.0 <= lat <= 85.0) and (-170.0 <= lon <= -30.0)

    keep = {aid for aid,(lat,lon,_) in airports.items() if in_na(lat,lon)}

    G = nx.Graph()
    G.add_nodes_from(keep)

    with open(routes_f, 'r', encoding='utf-8', errors='ignore') as f:
        rdr = csv.reader(f)
        for row in rdr:
            try:
                src_id = int(row[3]); dst_id = int(row[5])
            except Exception:
                continue
            if src_id in keep and dst_id in keep and src_id != dst_id:
                G.add_edge(src_id, dst_id)

    group_of = {aid: airports[aid][2] for aid in G.nodes() if aid in airports}
    # keep nodes with group
    nodes_ok = [u for u in G.nodes() if u in group_of]
    G = G.subgraph(nodes_ok).copy()
    group_of = {u: group_of[u] for u in G.nodes()}

    G, group_of = take_largest_cc(G, group_of)
    return G, group_of, 'OpenFlights-NA'

DATASETS = {
    'Cora': load_cora,
    'Email-Eu-core': load_email_eu_core,
    'Facebook': load_facebook,
    'CollegeMsg': load_collegemsg,
    'OpenFlights-NA': load_openflights_na,
}

def summarize_graph(G, group_of, name):
    groups = Counter(group_of.values())
    print(f'{name}: |V|={G.number_of_nodes():,} |E|={G.number_of_edges():,} |groups|={len(groups):,}')

# quick sanity load (does not run algorithms)
for nm, fn in DATASETS.items():
    try:
        G, group_of, name = fn()
        summarize_graph(G, group_of, name)
    except Exception as e:
        print(f'[skip] {nm}: {e}')

## 4) Independent Cascade (IC) Monte Carlo + fairness metrics (reporting)

In [ ]:

# ==== Monte-Carlo simulation (IC) for final reporting
def ic_mc_spread_and_groups(G: nx.Graph, seeds: Set[int], p: float, group_of: Dict[int,int], sims: int=1000):
    n = G.number_of_nodes()
    group_sizes = Counter(group_of.values())
    totals = 0.0
    group_hits = Counter()
    nodes = list(G.nodes())
    adj = {u: list(G.neighbors(u)) for u in nodes}

    for _ in range(sims):
        active = set(seeds)
        frontier = list(seeds)
        while frontier:
            new_front = []
            for u in frontier:
                for v in adj[u]:
                    if v not in active and np.random.rand() < p:
                        active.add(v)
                        new_front.append(v)
            frontier = new_front
        totals += len(active)
        hits_g = Counter(group_of[u] for u in active)
        for g, cnt in hits_g.items():
            group_hits[g] += cnt / group_sizes[g]

    spread_mean = totals / sims
    f_g = {g: (group_hits[g] / sims) for g in group_sizes}
    return spread_mean, f_g


In [ ]:
# Additional helper: compute fairness metrics from group fractions (same shape as report columns)
def fairness_from_fractions(f_g: Dict[int, float], parity_cap: float = 1.0, welfare_alpha: float = 0.5):
    vals = np.array(list(f_g.values()), dtype=float)
    if vals.size == 0:
        return dict(gini1m=1.0, maxmin=1.0, parity1=1.0, welfare=1.0)
    # 1 - Gini
    if vals.sum() <= 1e-12:
        gini1m = 1.0
    else:
        x = np.sort(vals)
        n = len(x)
        cum = np.cumsum(x)
        gini = (n + 1 - 2*(cum.sum()/x.sum()))/n
        gini1m = float(max(0.0, 1.0 - gini))
    maxmin = float(np.min(vals))
    mean = float(np.mean(vals))
    gap = float(np.max(np.abs(vals - mean)))
    cap = max(float(parity_cap), 1e-9)
    parity1 = float(max(0.0, 1.0 - gap / cap))
    welfare = float(np.mean(np.power(vals, welfare_alpha)))
    return dict(gini1m=gini1m, maxmin=maxmin, parity1=parity1, welfare=welfare)

def ic_mc_spread_only(G: nx.Graph, seeds: Set[int], p: float, sims: int = 1000):
    spread, _ = ic_mc_spread_and_groups(G, seeds, p, {u:0 for u in G.nodes()}, sims=sims)
    return spread

## 5) Reverse Influence Sampling (RIS) precomputation + RR evaluation

In [ ]:

# ==== Reverse Influence Sampling (RIS)
def _build_in_adj(adj_list: Dict[int, List[int]], directed=False):
    if not directed:
        return {u: list(neigh) for u, neigh in adj_list.items()}
    in_adj = {u: [] for u in adj_list}
    for u, nbrs in adj_list.items():
        for v in nbrs:
            in_adj[v].append(u)
    return in_adj

def _reverse_bfs_ic(start: int, in_adj: Dict[int, List[int]], p: float, rng: random.Random) -> Set[int]:
    visited = set([start])
    q = [start]
    while q:
        v = q.pop()
        for u in in_adj.get(v, []):
            if u not in visited and rng.random() < p:
                visited.add(u)
                q.append(u)
    return visited

@dataclass
class RRPack:
    RR_sets: List[List[int]]
    RR_root_group: List[int]
    node_to_RR: Dict[int, List[int]]
    theta_g: Dict[int, int]
    n: int
    groups: List[int]

def build_rr_sets_from_graph(G: nx.Graph, group_of: Dict[int,int], theta: int, p: float, seed=SEED) -> RRPack:
    rng = random.Random(seed)
    nodes = list(G.nodes())
    n = len(nodes)
    adj_list = build_adj_list(G)
    in_adj = _build_in_adj(adj_list, directed=False)

    RR_sets, RR_root_group = [], []
    node_to_RR = {u: [] for u in nodes}

    for rid in range(theta):
        v = rng.choice(nodes)
        R = _reverse_bfs_ic(v, in_adj, p, rng)
        RR_sets.append(list(R))
        g = group_of[v]
        RR_root_group.append(g)
        for u in R:
            node_to_RR[u].append(rid)

    theta_g = Counter(RR_root_group)
    groups = sorted(theta_g.keys())
    return RRPack(RR_sets, RR_root_group, node_to_RR, theta_g, n, groups)


In [ ]:

# ==== Fairness metrics (normalized to [0,1], higher is better)
def fairness_from_rr_cover(covered_rr_ids: Set[int], rr: RRPack, parity_cap: float = 1.0, welfare_alpha: float = 0.5):
    cov_g = Counter()
    for rid in covered_rr_ids:
        g = rr.RR_root_group[rid]
        cov_g[g] += 1
    f_g = {g: (cov_g[g] / rr.theta_g[g]) if rr.theta_g[g] > 0 else 0.0 for g in rr.theta_g}

    vals = np.array(list(f_g.values()), dtype=float)
    if vals.size == 0:
        gini1m = 1.0
    else:
        if vals.sum() <= 1e-12:
            gini1m = 1.0
        else:
            vals_sorted = np.sort(vals)
            n = len(vals_sorted)
            cum = np.cumsum(vals_sorted)
            gini = (n + 1 - 2 * (cum.sum() / vals_sorted.sum())) / n
            gini1m = float(max(0.0, 1.0 - gini))

    maxmin = float(np.min(list(f_g.values()))) if len(f_g) else 1.0
    if len(f_g):
        mean = float(np.mean(vals))
        gap = float(np.max(np.abs(vals - mean)))
        cap = max(float(parity_cap), 1e-9)
        parity1 = float(max(0.0, 1.0 - gap / cap))
    else:
        parity1 = 1.0
    welfare = float(np.mean(np.power(vals, welfare_alpha))) if len(f_g) else 1.0

    return dict(f_g=f_g, gini1m=gini1m, maxmin=maxmin, parity1=parity1, welfare=welfare)

def cover_rr_ids(seeds: Iterable[int], node_to_RR: Dict[int, List[int]]) -> Set[int]:
    covered = set()
    for u in seeds:
        covered.update(node_to_RR[u])
    return covered


In [ ]:

# ==== RIS-Greedy baseline
def ris_greedy(rr: RRPack, k: int) -> Set[int]:
    covered = set()
    seeds = set()
    remaining = set(rr.node_to_RR.keys())
    for _ in range(k):
        best_u, best_gain = None, -1
        for u in remaining:
            gain = 0
            for rid in rr.node_to_RR[u]:
                if rid not in covered:
                    gain += 1
            if gain > best_gain:
                best_gain, best_u = gain, u
        if best_u is None:
            break
        seeds.add(best_u)
        for rid in rr.node_to_RR[best_u]:
            covered.add(rid)
        remaining.remove(best_u)
    return seeds


## 6) Baselines (heuristics + RIS-Greedy)

In [ ]:

# ==== Heuristic baselines
def topk_degree(G: nx.Graph, k: int) -> Set[int]:
    deg = sorted(G.degree, key=lambda x: x[1], reverse=True)
    return set([u for u, _ in deg[:k]])

def topk_closeness(G: nx.Graph, k: int) -> Set[int]:
    clo = nx.closeness_centrality(G)
    return set(sorted(clo, key=lambda u: clo[u], reverse=True)[:k])

def topk_betweenness(G: nx.Graph, k: int) -> Set[int]:
    n = G.number_of_nodes()
    if n > 1500:
        bt = nx.betweenness_centrality(G, k=min(800, n), seed=SEED)
    else:
        bt = nx.betweenness_centrality(G)
    return set(sorted(bt, key=lambda u: bt[u], reverse=True)[:k])

def topk_pagerank(G: nx.Graph, k: int) -> Set[int]:
    pr = nx.pagerank(G, alpha=0.85, tol=1e-06)
    return set(sorted(pr, key=lambda u: pr[u], reverse=True)[:k])

def per_group_top_degree(G: nx.Graph, group_of: Dict[int,int], k: int) -> Set[int]:
    deg = dict(G.degree())
    best_per_g = {}
    for u, d in deg.items():
        g = group_of[u]
        if g not in best_per_g or d > best_per_g[g][1]:
            best_per_g[g] = (u, d)
    seeds = set(u for (u, _) in best_per_g.values())
    if len(seeds) > k:
        seeds = set([u for (u, d) in sorted(best_per_g.values(), key=lambda x: x[1], reverse=True)[:k]])
    elif len(seeds) < k:
        remaining = [u for u, d in sorted(deg.items(), key=lambda x: x[1], reverse=True) if u not in seeds]
        for u in remaining:
            seeds.add(u)
            if len(seeds) == k:
                break
    return seeds


## 7) Baseline: CELF++ (IC) — real implementation with progress

In [ ]:
# ==== CELF++ (IC) baseline (real; no stub) ====
def celfpp_select_seeds(
    G: nx.Graph,
    k: int,
    p: float,
    mc_select: int = 2000,
    base_seed: int = SEED,
    show_progress: bool = True,
    progress_every: int = 250
):
    """CELF++ for IC with MC oracle.
    Progress:
      - singleton evaluation over all nodes
      - greedy steps up to k with lazy stats
    Returns: (seed_list, time_seconds)
    """
    t0 = time.time()
    nodes = list(G.nodes())
    n = len(nodes)
    adj = {u: list(G.neighbors(u)) for u in nodes}

    sigma_cache = {}

    def sigma(seedset: Set[int]):
        key = frozenset(seedset)
        if key in sigma_cache:
            return sigma_cache[key]
        # stable RNG per seedset to reduce drift
        h = (hash((base_seed, tuple(sorted(key)))) & 0xFFFFFFFF)
        np.random.seed(h)
        val, _ = ic_mc_spread_and_groups(G, set(key), p, {u:0 for u in G.nodes()}, sims=mc_select)
        sigma_cache[key] = val
        return val

    tqdm = _get_tqdm()
    use_bar = show_progress and (tqdm is not None)

    heap = []
    mg1, prev_best, mg2, flag = {}, {}, {}, {}

    best_u, best_val = None, -1.0

    it = tqdm(nodes, desc=f"CELF++ singletons (p={p}, mc={mc_select})", total=n) if use_bar else nodes
    for i, u in enumerate(it, start=1):
        s_u = sigma({u})
        mg1[u] = s_u
        prev_best[u] = None
        mg2[u] = s_u
        flag[u] = 0
        heapq.heappush(heap, (-mg1[u], u))
        if s_u > best_val:
            best_val = s_u
            best_u = u
        if (not use_bar) and show_progress and (i % progress_every == 0):
            print(f"[CELF++] singletons: {i}/{n} done")

    S = [best_u]
    chosen = {best_u}
    last_seed = best_u
    if show_progress:
        print(f"[CELF++] picked seed #1 = {best_u} (sigma={best_val:.3f})")

    outer = tqdm(total=(k-1), desc="CELF++ greedy steps", leave=True) if use_bar else None

    while len(S) < k:
        sigma_S = sigma(set(S))
        cur_best, cur_best_gain = None, -1.0

        pops = recomputes = reused_mg2 = 0

        while True:
            neg_gain, u = heapq.heappop(heap)
            pops += 1
            if u in chosen:
                continue
            if abs((-neg_gain) - mg1.get(u, -1e18)) > 1e-12:
                continue

            if flag[u] == len(S):
                S.append(u)
                chosen.add(u)
                last_seed = u
                break

            if prev_best[u] == last_seed:
                mg1[u] = mg2[u]
                reused_mg2 += 1
                flag[u] = len(S)
                if mg1[u] > cur_best_gain:
                    cur_best_gain = mg1[u]
                    cur_best = u
            else:
                recomputes += 1
                sigma_Su = sigma(set(S) | {u})
                mg1[u] = sigma_Su - sigma_S
                prev_best[u] = cur_best

                if cur_best is None:
                    mg2[u] = mg1[u]
                else:
                    sigma_Sb = sigma(set(S) | {cur_best})
                    sigma_Sbu = sigma(set(S) | {cur_best, u})
                    mg2[u] = sigma_Sbu - sigma_Sb

                flag[u] = len(S)
                if mg1[u] > cur_best_gain:
                    cur_best_gain = mg1[u]
                    cur_best = u

            heapq.heappush(heap, (-mg1[u], u))

        if outer is not None:
            outer.update(1)
            outer.set_postfix({"picked": len(S), "pops": pops, "recomp": recomputes, "reuse": reused_mg2})
        elif show_progress:
            print(f"[CELF++] picked seed #{len(S)}/{k}={S[-1]} | pops={pops}, recomp={recomputes}, reuse={reused_mg2}")

    if outer is not None:
        outer.close()

    return S, (time.time() - t0)

def run_celfpp_one(G, group_of, name, k=20, p=0.10, mc_select=2000, mc_report=2000):
    seeds, t_sel = celfpp_select_seeds(G, k=k, p=p, mc_select=mc_select, show_progress=True)
    t0 = time.time()
    spread, f_g = ic_mc_spread_and_groups(G, set(seeds), p, group_of, sims=mc_report)
    fair = fairness_from_fractions(f_g, parity_cap=1.0, welfare_alpha=0.5)
    t_eval = time.time() - t0
    return dict(dataset=name, method='CELF++ (IC)', k=k, p=p,
                mc_select=mc_select, mc_report=mc_report,
                time_select_s=t_sel, time_eval_s=t_eval,
                spread=spread,
                one_minus_gini=fair['gini1m'],
                maxmin=fair['maxmin'],
                one_minus_parity=fair['parity1'],
                welfare=fair['welfare'],
                seeds=seeds)

## 8) FIMMOGA++ (proposed): RIS-in-GA, 5 objectives, NSGA-III

In [ ]:

# ==== GA helpers: evaluate, NSGA-II/III, operators, priors
def gini1m_from_fractions(f_g: Dict[int,float]) -> float:
    vals = np.array(list(f_g.values()), dtype=float)
    if vals.size == 0 or vals.sum() <= 1e-12:
        return 1.0
    vals.sort()
    n = len(vals)
    cum = np.cumsum(vals)
    gini = (n + 1 - 2 * (cum.sum() / vals.sum())) / n
    return float(max(0.0, 1.0 - gini))

def parity1_from_fractions(f_g: Dict[int,float], cap=1.0) -> float:
    vals = np.array(list(f_g.values()), dtype=float)
    if vals.size == 0:
        return 1.0
    mean = float(vals.mean())
    gap = float(np.max(np.abs(vals - mean)))
    return float(max(0.0, 1.0 - gap / max(cap,1e-9)))

def welfare_from_fractions(f_g: Dict[int,float], alpha=0.5) -> float:
    vals = np.array(list(f_g.values()), dtype=float)
    return float(np.mean(np.power(vals, alpha))) if vals.size>0 else 1.0

def evaluate_seed_rr_multi(seeds: Set[int], rr: RRPack, parity_cap=1.0, welfare_alpha=0.5, spread_norm_by=None):
    covered = cover_rr_ids(seeds, rr.node_to_RR)
    est_spread = rr.n * (len(covered) / max(1, len(rr.RR_sets)))
    f = fairness_from_rr_cover(covered, rr, parity_cap=parity_cap, welfare_alpha=welfare_alpha)
    spread_norm = est_spread / (rr.n if spread_norm_by is None else max(spread_norm_by,1e-9))
    spread_norm = float(min(1.0, max(0.0, spread_norm)))
    obj = (spread_norm, f["gini1m"], f["maxmin"], f["parity1"], f["welfare"])
    meta = dict(spread=est_spread, f_g=f["f_g"])
    return obj, meta

def nd_sort(F: List[List[float]]):
    N = len(F)
    S = [set() for _ in range(N)]
    n_dom = [0]*N
    rank = [0]*N
    fronts = [[]]
    M = len(F[0])

    for p in range(N):
        for q in range(N):
            if p==q: continue
            if all(F[p][m] >= F[q][m] for m in range(M)) and any(F[p][m] > F[q][m] for m in range(M)):
                S[p].add(q)
            elif all(F[q][m] >= F[p][m] for m in range(M)) and any(F[q][m] > F[p][m] for m in range(M)):
                n_dom[p] += 1
        if n_dom[p] == 0:
            rank[p] = 1
            fronts[0].append(p)
    i = 0
    while fronts[i]:
        Q = []
        for p in fronts[i]:
            for q in S[p]:
                n_dom[q] -= 1
                if n_dom[q] == 0:
                    rank[q] = i + 2
                    Q.append(q)
        i += 1
        fronts.append(Q)
    if not fronts[-1]:
        fronts.pop()
    return fronts, rank

def crowding(front_idx: List[int], F: List[List[float]]) -> Dict[int,float]:
    if not front_idx:
        return {}
    M = len(F[0])
    dist = {i: 0.0 for i in front_idx}
    for m in range(M):
        sorted_idx = sorted(front_idx, key=lambda i: F[i][m])
        fmin, fmax = F[sorted_idx[0]][m], F[sorted_idx[-1]][m]
        dist[sorted_idx[0]] = float('inf')
        dist[sorted_idx[-1]] = float('inf')
        if abs(fmax - fmin) < 1e-12: continue
        for j in range(1, len(sorted_idx)-1):
            i_prev, i_next = sorted_idx[j-1], sorted_idx[j+1]
            dist[sorted_idx[j]] += (F[i_next][m] - F[i_prev][m]) / (fmax - fmin)
    return dist

def das_dennis_points(M, p=12):
    out = []
    def rec(gen, left, acc):
        if gen == M-1:
            out.append(np.array(acc + [left], float) / p)
        else:
            for i in range(left+1):
                rec(gen+1, left-i, acc+[i])
    rec(0, p, [])
    return out

def normalize_F(F: List[List[float]]):
    A = np.array(F, float)
    mins = A.min(axis=0)
    maxs = A.max(axis=0)
    denom = np.maximum(maxs - mins, 1e-12)
    return (A - mins)/denom, mins, maxs

def associate_to_refs(front_idx: List[int], F_norm: np.ndarray, refs: List[np.ndarray]):
    assoc = {j: [] for j in range(len(refs))}
    closest = {}
    for i in front_idx:
        y = F_norm[i]
        yn = y / (np.linalg.norm(y) + 1e-12)
        best_j, best_d = None, float('inf')
        for j, r in enumerate(refs):
            rn = r / (np.linalg.norm(r) + 1e-12)
            cos = float(np.dot(yn, rn))
            d = math.sqrt(max(0.0, 1.0 - cos*cos))
            if d < best_d:
                best_d, best_j = d, j
        assoc[best_j].append((i, best_d))
        closest[i] = best_j
    return assoc, closest

def nsga3_select(pop, F, pop_size, ref_p=12):
    fronts, _ = nd_sort(F)
    new_pop, new_F = [], []
    F_norm, *_ = normalize_F(F)
    refs = das_dennis_points(len(F[0]), p=ref_p)
    niche_count = [0]*len(refs)

    for fr in fronts:
        if len(new_pop) + len(fr) <= pop_size:
            for idx in fr:
                new_pop.append(pop[idx]); new_F.append(F[idx])
                _, close = associate_to_refs([idx], F_norm, refs)
                niche_count[close[idx]] += 1
        else:
            assoc, _ = associate_to_refs(fr, F_norm, refs)
            need = pop_size - len(new_pop)
            chosen = set()
            while need > 0:
                j = int(np.argmin(niche_count))
                candidates = [(i,d) for (i,d) in assoc.get(j, []) if i not in chosen]
                if not candidates:
                    niche_count[j] = float('inf')
                    if all(x == float('inf') for x in niche_count):
                        break
                    continue
                i_sel = min(candidates, key=lambda t: t[1])[0]
                chosen.add(i_sel)
                new_pop.append(pop[i_sel]); new_F.append(F[i_sel])
                niche_count[j] += 1
                need -= 1
            break
    return new_pop, new_F

def mutate_comm_aware(S: Set[int], group_of: Dict[int,int], nodes_by_group: Dict[int,List[int]], rr: RRPack, rate=0.30):
    if random.random() > rate or not S:
        return S
    S = set(S)
    u = random.choice(list(S))
    g = group_of[u]
    cand = [v for v in nodes_by_group[g] if v not in S and v != u]
    if not cand: return S
    covered = cover_rr_ids(S, rr.node_to_RR)
    def rr_gain(x):
        return sum(1 for rid in rr.node_to_RR[x] if rid not in covered)
    v = max(cand, key=rr_gain)
    S.remove(u); S.add(v)
    return S

def local_improve_one_swap_rr(S: Set[int], rr: RRPack, nodes: List[int], iters=1):
    S = set(S)
    for _ in range(iters):
        covered = cover_rr_ids(S, rr.node_to_RR)
        u = random.choice(list(S))
        best_v, best_gain = None, -1
        for v in nodes:
            if v in S: continue
            gain = sum(1 for rid in rr.node_to_RR[v] if rid not in covered)
            if gain > best_gain:
                best_gain, best_v = gain, v
        if best_v is not None:
            S.remove(u); S.add(best_v)
    return S

def build_prior_population(G: nx.Graph, group_of: Dict[int,int], k: int, pop_size: int) -> List[Set[int]]:
    pop = []
    pop.append(topk_degree(G, k))
    pop.append(topk_closeness(G, k))
    pop.append(topk_betweenness(G, k))
    pop.append(topk_pagerank(G, k))
    pop.append(per_group_top_degree(G, group_of, k))
    nodes = list(G.nodes())
    while len(pop) < pop_size:
        pop.append(set(random.sample(nodes, k)))
    return pop

def crossover_sets(p1: Set[int], p2: Set[int], k: int, nodes: List[int]) -> Set[int]:
    child = set(random.sample(list(p1), min(k//2, len(p1))))
    for x in p2:
        if len(child) >= k: break
        if x not in child: child.add(x)
    while len(child) < k:
        c = random.choice(nodes)
        if c not in child: child.add(c)
    return child


In [ ]:

# ==== FIMMOGA++ driver
def fimmoga_rr(G: nx.Graph, group_of: Dict[int,int], rr: RRPack, k: int,
               pop_size=60, generations=80, mutation_rate=0.30,
               use_nsga3=True, nsga3_ref_p=12,
               parity_cap=1.0, welfare_alpha=0.5,
               normalize_spread_by=None,
               local_search_every=5, local_search_iters=1):
    set_seed(SEED)
    nodes = list(G.nodes())
    nodes_by_group = defaultdict(list)
    for u,g in group_of.items():
        nodes_by_group[g].append(u)
    # init
    population = build_prior_population(G, group_of, k, pop_size)
    F, MET = [], []
    for S in population:
        obj, meta = evaluate_seed_rr_multi(S, rr, parity_cap, welfare_alpha, normalize_spread_by)
        F.append(obj); MET.append(meta)

    M = len(F[0])
    for gen in range(generations):
        fronts, ranks = nd_sort(F)
        crowd_map = {}
        for fr in fronts:
            crowd_map.update(crowding(fr, F))
        def pick():
            i, j = random.sample(range(len(population)), 2)
            if ranks[i] < ranks[j]: return i
            if ranks[j] < ranks[i]: return j
            if M <= 3 or not use_nsga3:
                return i if crowd_map.get(i,0.0) > crowd_map.get(j,0.0) else j
            else:
                return i if sum(F[i]) > sum(F[j]) else j
        parents = [population[pick()] for _ in range(len(population))]
        offspring = []
        for i in range(0, len(parents), 2):
            p1 = parents[i]
            p2 = parents[(i+1) % len(parents)]
            c1 = crossover_sets(p1, p2, k, nodes)
            if random.random() < mutation_rate:
                c1 = mutate_comm_aware(c1, group_of, nodes_by_group, rr, rate=1.0)
            offspring.append(c1)
        if local_search_every and (gen+1) % local_search_every == 0:
            offspring = [local_improve_one_swap_rr(S, rr, nodes, iters=local_search_iters) for S in offspring]
        F_off, M_off = [], []
        for S in offspring:
            obj, meta = evaluate_seed_rr_multi(S, rr, parity_cap, welfare_alpha, normalize_spread_by)
            F_off.append(obj); M_off.append(meta)
        combined_pop = population + offspring
        combined_F = F + F_off
        if M >= 4 and use_nsga3:
            population, F = nsga3_select(combined_pop, combined_F, pop_size, ref_p=nsga3_ref_p)
        else:
            fronts, _ = nd_sort(combined_F)
            new_pop, new_F = [], []
            for fr in fronts:
                if len(new_pop) + len(fr) <= pop_size:
                    for idx in fr:
                        new_pop.append(combined_pop[idx]); new_F.append(combined_F[idx])
                else:
                    cd = crowding(fr, combined_F)
                    sorted_fr = sorted(fr, key=lambda i: cd[i], reverse=True)
                    need = pop_size - len(new_pop)
                    for idx in sorted_fr[:need]:
                        new_pop.append(combined_pop[idx]); new_F.append(combined_F[idx])
                    break
            population, F = new_pop, new_F
        if (gen+1) % 5 == 0 or gen == 0:
            spreads = [evaluate_seed_rr_multi(S, rr)[1]["spread"] for S in population]
            print(f"[GA-RIS] Gen {gen+1}/{generations} | spread=({min(spreads):.1f},{max(spreads):.1f})")
    return population, F


In [ ]:
# Helper: run FIMMOGA++ once and choose a single representative solution for reporting
def run_fimmogapp_one(G, group_of, name, rr: RRPack, k=20, p=0.10, pop_size=60, generations=60, mc_report=2000):
    t0 = time.time()
    pop, F = fimmoga_rr(
        G, group_of, rr, k,
        pop_size=pop_size, generations=generations,
        mutation_rate=0.30,
        use_nsga3=True, nsga3_ref_p=12,
        parity_cap=1.0, welfare_alpha=0.5,
        normalize_spread_by=rr.n,
        local_search_every=5, local_search_iters=1
    )
    t_sel = time.time() - t0
    fronts, _ = nd_sort(F)
    pf = fronts[0]
    # pick by sum of fairness-like objectives (exclude spread at index 0)
    idx_best = max(pf, key=lambda i: sum(F[i][1:]))
    seeds = pop[idx_best]

    t1 = time.time()
    spread, f_g = ic_mc_spread_and_groups(G, set(seeds), p, group_of, sims=mc_report)
    fair = fairness_from_fractions(f_g, parity_cap=1.0, welfare_alpha=0.5)
    t_eval = time.time() - t1

    return dict(dataset=name, method='FIMMOGA++ (RIS-in-GA)', k=k, p=p,
                theta=sum(rr.theta_g.values()),
                pop_size=pop_size, generations=generations,
                mc_report=mc_report,
                time_select_s=t_sel, time_eval_s=t_eval,
                spread=spread,
                one_minus_gini=fair['gini1m'],
                maxmin=fair['maxmin'],
                one_minus_parity=fair['parity1'],
                welfare=fair['welfare'],
                seeds=list(seeds))

## 9) FIMMOGA (orig): MC-in-GA, 2 objectives, NSGA-II (real)

In [ ]:
# ==== FIMMOGA (orig): MC-in-GA, 2 objectives (Spread, 1-Gini), NSGA-II ====

def evaluate_seed_mc_twoobj(G: nx.Graph, group_of: Dict[int,int], seeds: Iterable[int],
                            p: float, mc_sims: int, base_seed: int = SEED):
    """Returns: (F=[spread_norm, gini1m], spread_raw, f_g)"""
    key = tuple(sorted(seeds))
    # stable per seedset
    h = (hash((base_seed, key, p, mc_sims)) & 0xFFFFFFFF)
    np.random.seed(h)
    spread, f_g = ic_mc_spread_and_groups(G, set(key), p, group_of, sims=mc_sims)
    fair = fairness_from_fractions(f_g, parity_cap=1.0, welfare_alpha=0.5)
    spread_norm = spread / max(1, G.number_of_nodes())
    return [float(spread_norm), float(fair['gini1m'])], float(spread), f_g

def mutate_group_swap_mc(S: Set[int], group_of: Dict[int,int], nodes_by_group: Dict[int,List[int]], rate=0.30):
    if random.random() > rate or not S:
        return S
    S = set(S)
    u = random.choice(list(S))
    g = group_of[u]
    cand = [v for v in nodes_by_group.get(g, []) if v not in S and v != u]
    if not cand:
        # fallback: any node
        return S
    # pick high-degree within group (cheap heuristic) if degrees present globally later
    v = random.choice(cand)
    S.remove(u); S.add(v)
    return S

def fimmoga_mc(G: nx.Graph, group_of: Dict[int,int], k: int, p: float,
              pop_size=60, generations=60, mutation_rate=0.30,
              mc_ga=200,  # MC sims INSIDE GA
              base_seed: int = SEED):
    """Returns (population, F) where F are 2-objective vectors [spread_norm, gini1m]."""
    set_seed(base_seed)
    nodes = list(G.nodes())
    nodes_by_group = defaultdict(list)
    for u,g in group_of.items():
        nodes_by_group[g].append(u)

    # use the same prior init helper used by FIMMOGA++ (degree/closeness/between/pagerank/per-group/random)
    population = build_prior_population(G, group_of, k, pop_size)
    population = [set(ind) if not isinstance(ind, set) else ind for ind in population]

    # cache to reduce repeated MC calls
    cache = {}

    def eval_ind(S):
        key = tuple(sorted(S))
        if key in cache:
            return cache[key]
        f, _, _ = evaluate_seed_mc_twoobj(G, group_of, key, p, mc_ga, base_seed=base_seed)
        cache[key] = f
        return f

    F = [eval_ind(S) for S in population]

    tqdm = _get_tqdm()
    gen_iter = tqdm(range(generations), desc=f'GA-MC (orig) {k=},{p=}', total=generations) if tqdm else range(generations)

    for gen in gen_iter:
        fronts, ranks = nd_sort(F)
        crowd_map = {}
        for fr in fronts:
            crowd_map.update(crowding(fr, F))

        def pick_idx():
            i, j = random.sample(range(len(population)), 2)
            if ranks[i] < ranks[j]: return i
            if ranks[j] < ranks[i]: return j
            return i if crowd_map.get(i,0.0) > crowd_map.get(j,0.0) else j

        parents = [population[pick_idx()] for _ in range(len(population))]
        offspring = []
        for i in range(0, len(parents), 2):
            A = parents[i]
            B = parents[i+1] if i+1 < len(parents) else parents[i]
            child1 = crossover_sets(A, B, k, nodes)
            child2 = crossover_sets(B, A, k, nodes)
            child1 = mutate_group_swap_mc(child1, group_of, nodes_by_group, rate=mutation_rate)
            child2 = mutate_group_swap_mc(child2, group_of, nodes_by_group, rate=mutation_rate)
            offspring.append(child1); offspring.append(child2)

        offspring = offspring[:pop_size]
        off_F = [eval_ind(S) for S in offspring]

        combined_pop = population + offspring
        combined_F = F + off_F

        # NSGA-II environmental selection (M=2, so we can use crowding)
        fronts, _ = nd_sort(combined_F)
        new_pop, new_F = [], []
        for fr in fronts:
            if len(new_pop) + len(fr) <= pop_size:
                for idx in fr:
                    new_pop.append(combined_pop[idx]); new_F.append(combined_F[idx])
            else:
                cd = crowding(fr, combined_F)
                sorted_fr = sorted(fr, key=lambda i: cd[i], reverse=True)
                need = pop_size - len(new_pop)
                for idx in sorted_fr[:need]:
                    new_pop.append(combined_pop[idx]); new_F.append(combined_F[idx])
                break
        population, F = new_pop, new_F

        if tqdm:
            fronts_now, _ = nd_sort(F)
            nd = fronts_now[0]
            gen_iter.set_postfix({'ND': len(nd), 'best_sum': f'{max(sum(F[i]) for i in nd):.3f}'})

    return population, F

def run_fimmoga_orig_one(G, group_of, name, k=20, p=0.10,
                         pop_size=60, generations=60,
                         mc_ga=200, mc_report=2000):
    t0 = time.time()
    pop, F = fimmoga_mc(G, group_of, k=k, p=p, pop_size=pop_size, generations=generations, mc_ga=mc_ga)
    t_sel = time.time() - t0

    fronts, _ = nd_sort(F)
    pf = fronts[0]
    # balanced pick: maximize average of two objectives
    idx = max(pf, key=lambda i: (F[i][0] + F[i][1]) / 2.0)
    seeds = pop[idx]

    t1 = time.time()
    spread, f_g = ic_mc_spread_and_groups(G, set(seeds), p, group_of, sims=mc_report)
    fair = fairness_from_fractions(f_g, parity_cap=1.0, welfare_alpha=0.5)
    t_eval = time.time() - t1

    return dict(dataset=name, method='FIMMOGA (orig, MC-in-GA)', k=k, p=p,
                pop_size=pop_size, generations=generations,
                mc_ga=mc_ga, mc_report=mc_report,
                time_select_s=t_sel, time_eval_s=t_eval,
                spread=spread,
                one_minus_gini=fair['gini1m'],
                maxmin=fair['maxmin'],
                one_minus_parity=fair['parity1'],
                welfare=fair['welfare'],
                seeds=list(seeds))

## 10) Experiment runners (match the report tables)

In [ ]:
# ==== Experiment runners (Table-style outputs) ====

def build_rr_for_dataset(G, group_of, p=0.10, theta=100_000):
    return build_rr_sets_from_graph(G, group_of, theta=theta, p=p, seed=SEED)

def run_table3_overall_one_dataset(
    dataset_name: str,
    k: int = 20,
    p: float = 0.10,
    theta: int = 100_000,
    pop_size: int = 60,
    generations: int = 60,
    mc_report: int = 2000,
    celf_mc_select: int = 2000,
    fimmoga_mc_ga: int = 200,
    include_celf: bool = True,
):
    """Reproduces the *format* of Table 3 for a single dataset at k=20, p=0.10.
    Returns a DataFrame with one row per method.
    """
    G, group_of, NAME = DATASETS[dataset_name]()
    n, m = G.number_of_nodes(), G.number_of_edges()
    print(f"\n=== {NAME} | n={n:,}, m={m:,} | k={k}, p={p} ===")

    rr = build_rr_for_dataset(G, group_of, p=p, theta=theta)

    rows = []

    # heuristic baselines (selection cost negligible here)
    baseline_methods = {
        'Degree': topk_degree(G, k),
        'Closeness': topk_closeness(G, k),
        'Betweenness': topk_betweenness(G, k),
        'PageRank': topk_pagerank(G, k),
        'Per-Group Top-Degree': per_group_top_degree(G, group_of, k),
        'RIS-Greedy': ris_greedy(rr, k),
    }

    for meth, seeds in baseline_methods.items():
        t0 = time.time()
        spread, f_g = ic_mc_spread_and_groups(G, set(seeds), p, group_of, sims=mc_report)
        fair = fairness_from_fractions(f_g, parity_cap=1.0, welfare_alpha=0.5)
        t_eval = time.time() - t0
        rows.append(dict(dataset=NAME, method=meth, k=k, p=p,
                         time_select_s=0.0, time_eval_s=t_eval,
                         spread=spread,
                         one_minus_gini=fair['gini1m'],
                         maxmin=fair['maxmin'],
                         one_minus_parity=fair['parity1'],
                         welfare=fair['welfare'],
                         seeds=list(seeds)))

    # CELF++ baseline
    if include_celf:
        rows.append(run_celfpp_one(G, group_of, NAME, k=k, p=p, mc_select=celf_mc_select, mc_report=mc_report))

    # FIMMOGA (orig) — MC-in-GA
    rows.append(run_fimmoga_orig_one(G, group_of, NAME, k=k, p=p,
                                     pop_size=pop_size, generations=generations,
                                     mc_ga=fimmoga_mc_ga, mc_report=mc_report))

    # FIMMOGA++ — RIS-in-GA
    rows.append(run_fimmogapp_one(G, group_of, NAME, rr, k=k, p=p,
                                  pop_size=pop_size, generations=generations,
                                  mc_report=mc_report))

    df = pd.DataFrame(rows)
    out = RESULTS_DIR / f"Table3_like_{NAME}_k{k}_p{p}.csv"
    df.to_csv(out, index=False)
    print("Saved:", out)
    return df

def run_table7_mc_vs_ris_one_dataset(
    dataset_name: str,
    k: int = 20,
    p: float = 0.10,
    theta: int = 100_000,
    pop_size: int = 60,
    generations: int = 60,
    mc_report: int = 2000,
    mc_ga: int = 200,
):
    """Reproduces Table 7 idea: MC-in-GA (orig) vs RIS-in-GA (ours), at k=20, p=0.10."""
    G, group_of, NAME = DATASETS[dataset_name]()
    rr = build_rr_for_dataset(G, group_of, p=p, theta=theta)

    # MC-in-GA
    df_mc = run_fimmoga_orig_one(G, group_of, NAME, k=k, p=p,
                                 pop_size=pop_size, generations=generations,
                                 mc_ga=mc_ga, mc_report=mc_report)

    # RIS-in-GA
    df_ris = run_fimmogapp_one(G, group_of, NAME, rr, k=k, p=p,
                               pop_size=pop_size, generations=generations,
                               mc_report=mc_report)

    df = pd.DataFrame([df_mc, df_ris])
    out = RESULTS_DIR / f"Table7_like_MC_vs_RIS_{NAME}_k{k}_p{p}.csv"
    df.to_csv(out, index=False)
    print("Saved:", out)
    return df

## 10b) Sensitivity sweeps and ablations (report sections)

These functions mirror the report-style experiments:
- Sensitivity to **k** and **p**
- Ablation: **MC-in-GA vs RIS-in-GA** (Table 7-style)
- Ablation: **NSGA-II vs NSGA-III** inside RIS GA (Table 8-style)

They are written to run **one dataset at a time** and save CSVs into `./results/`.


In [ ]:
# ==== Sensitivity sweeps + ablations ====

def evaluate_seed_on_rr_and_mc(G, group_of, rr, seeds, p, mc_eval_report):
    # RIS-based estimate + RR-based fairness
    obj, meta = evaluate_seed_rr_multi(seeds, rr, parity_cap=1.0, welfare_alpha=0.5, spread_norm_by=rr.n)
    spread_ris_est = meta["spread"]
    fairness_rr = fairness_from_rr_cover(cover_rr_ids(seeds, rr.node_to_RR), rr)

    # Final MC evaluation for report numbers
    spread_mc, f_g_mc = ic_mc_spread_and_groups(G, set(seeds), p, group_of, sims=mc_eval_report)
    fair_mc = fairness_from_fractions(f_g_mc, parity_cap=1.0, welfare_alpha=0.5)

    row = dict(
        spread_ris_estimate=float(spread_ris_est),
        spread_mc=float(spread_mc),
        gini1m=float(fair_mc["gini1m"]),
        maxmin=float(fair_mc["maxmin"]),
        parity1=float(fair_mc["parity1"]),
        welfare=float(fair_mc["welfare"]),
        # RR fairness (optional debug)
        gini1m_rr=float(gini1m_from_fractions(fairness_rr["f_g"])),
        maxmin_rr=float(fairness_rr["maxmin"]),
        parity1_rr=float(fairness_rr["parity1"]),
        welfare_rr=float(fairness_rr["welfare"]),
    )
    return row

def sensitivity_k_one_dataset(dataset_name: str, p: float = 0.10, theta: int = 100_000, ks=(10,20,30),
                              mc_eval_report: int = 2000):
    G, group_of, NAME = DATASETS[dataset_name]()
    rr = build_rr_sets_from_graph(G, group_of, theta=theta, p=p, seed=SEED)

    rows = []
    for k in ks:
        seeds = ris_greedy(rr, k)
        stats = evaluate_seed_on_rr_and_mc(G, group_of, rr, seeds, p, mc_eval_report)
        rows.append(dict(dataset=NAME, k=k, p=p, theta=theta, method="RIS-Greedy", **stats))

    df = pd.DataFrame(rows)
    out = RESULTS_DIR / f"sensitivity_k_{NAME}_p{p}_theta{theta}.csv"
    df.to_csv(out, index=False)
    print("Saved:", out)
    return df

def sensitivity_p_one_dataset(dataset_name: str, k: int = 20, thetas=(50_000,100_000,200_000),
                              ps=(0.05,0.10,0.20), mc_eval_report: int = 2000):
    G, group_of, NAME = DATASETS[dataset_name]()
    rows = []
    for p in ps:
        for theta in thetas:
            rr = build_rr_sets_from_graph(G, group_of, theta=theta, p=p, seed=SEED)
            seeds = ris_greedy(rr, k)
            stats = evaluate_seed_on_rr_and_mc(G, group_of, rr, seeds, p, mc_eval_report)
            rows.append(dict(dataset=NAME, k=k, p=p, theta=theta, method="RIS-Greedy", **stats))

    df = pd.DataFrame(rows)
    out = RESULTS_DIR / f"sensitivity_p_{NAME}_k{k}.csv"
    df.to_csv(out, index=False)
    print("Saved:", out)
    return df

def ablation_fitness_engine_one_dataset(dataset_name: str, k: int = 20, p: float = 0.10, theta: int = 100_000,
                                        pop_size: int = 60, generations: int = 60,
                                        mc_ga: int = 200, mc_eval_report: int = 2000):
    """Table 7-style: MC-in-GA vs RIS-in-GA (ours)."""
    G, group_of, NAME = DATASETS[dataset_name]()
    rr = build_rr_sets_from_graph(G, group_of, theta=theta, p=p, seed=SEED)

    r_mc = run_fimmoga_orig_one(G, group_of, NAME, k=k, p=p,
                                pop_size=pop_size, generations=generations,
                                mc_ga=mc_ga, mc_report=mc_eval_report)

    r_ris = run_fimmogapp_one(G, group_of, NAME, rr, k=k, p=p,
                               pop_size=pop_size, generations=generations,
                               mc_report=mc_eval_report)

    df = pd.DataFrame([
        dict(dataset=NAME, fitness_in_GA="MC (FIMMOGA orig)", GA_time_s=r_mc["time_select_s"],
             Spread=r_mc["spread"], Gini1m=r_mc["one_minus_gini"], MaxMin=r_mc["maxmin"],
             Parity1=r_mc["one_minus_parity"], Welfare=r_mc["welfare"]),
        dict(dataset=NAME, fitness_in_GA="RIS (FIMMOGA++)", GA_time_s=r_ris["time_select_s"],
             Spread=r_ris["spread"], Gini1m=r_ris["one_minus_gini"], MaxMin=r_ris["maxmin"],
             Parity1=r_ris["one_minus_parity"], Welfare=r_ris["welfare"]),
    ])
    out = RESULTS_DIR / f"ablation_fitness_MC_vs_RIS_{NAME}_k{k}_p{p}.csv"
    df.to_csv(out, index=False)
    print("Saved:", out)
    return df

def run_fimmogapp_one_custom(G, group_of, name, rr: RRPack, k=20, p=0.10,
                            pop_size=60, generations=60, mc_report=2000,
                            use_nsga3=True):
    t0 = time.time()
    pop, F = fimmoga_rr(
        G, group_of, rr, k,
        pop_size=pop_size, generations=generations,
        mutation_rate=0.30,
        use_nsga3=use_nsga3, nsga3_ref_p=12,
        parity_cap=1.0, welfare_alpha=0.5,
        normalize_spread_by=rr.n,
        local_search_every=5, local_search_iters=1
    )
    t_sel = time.time() - t0

    fronts, _ = nd_sort(F)
    pf = fronts[0]
    idx_best = max(pf, key=lambda i: sum(F[i][1:]))
    seeds = pop[idx_best]

    t1 = time.time()
    spread, f_g = ic_mc_spread_and_groups(G, set(seeds), p, group_of, sims=mc_report)
    fair = fairness_from_fractions(f_g, parity_cap=1.0, welfare_alpha=0.5)
    t_eval = time.time() - t1

    return dict(dataset=name, method=('FIMMOGA++ (NSGA-III)' if use_nsga3 else 'FIMMOGA++ (NSGA-II)'),
                k=k, p=p, theta=sum(rr.theta_g.values()),
                pop_size=pop_size, generations=generations,
                mc_report=mc_report,
                time_select_s=t_sel, time_eval_s=t_eval,
                spread=spread,
                one_minus_gini=fair['gini1m'],
                maxmin=fair['maxmin'],
                one_minus_parity=fair['parity1'],
                welfare=fair['welfare'],
                seeds=list(seeds))

def ablation_nsga_one_dataset(dataset_name: str, k: int = 20, p: float = 0.10, theta: int = 100_000,
                             pop_size: int = 60, generations: int = 60, mc_eval_report: int = 2000):
    """Table 8-style: NSGA-II vs NSGA-III for RIS-based GA."""
    G, group_of, NAME = DATASETS[dataset_name]()
    rr = build_rr_sets_from_graph(G, group_of, theta=theta, p=p, seed=SEED)

    r2 = run_fimmogapp_one_custom(G, group_of, NAME, rr, k=k, p=p, pop_size=pop_size,
                                 generations=generations, mc_report=mc_eval_report, use_nsga3=False)
    r3 = run_fimmogapp_one_custom(G, group_of, NAME, rr, k=k, p=p, pop_size=pop_size,
                                 generations=generations, mc_report=mc_eval_report, use_nsga3=True)

    df = pd.DataFrame([r2, r3])
    out = RESULTS_DIR / f"ablation_nsga_{NAME}_k{k}_p{p}.csv"
    df.to_csv(out, index=False)
    print("Saved:", out)
    return df


## 11) Run one dataset at a time (recommended)

In [ ]:
# Example runs (uncomment ONE dataset at a time)

# --- Table 3-like overall comparison (k=20, p=0.10) ---
# df = run_table3_overall_one_dataset('Cora', include_celf=True)
# display(df)

# df = run_table3_overall_one_dataset('Email-Eu-core', include_celf=True)
# display(df)

# df = run_table3_overall_one_dataset('Facebook', include_celf=True)
# display(df)

# df = run_table3_overall_one_dataset('CollegeMsg', include_celf=True)
# display(df)

# df = run_table3_overall_one_dataset('OpenFlights-NA', include_celf=True)
# display(df)

# --- Table 7-like ablation: MC-in-GA vs RIS-in-GA ---
# df7 = run_table7_mc_vs_ris_one_dataset('Cora')
# display(df7)
